# 기상청 API를 이용한 서울시 체감온도(ta_chi) 데이터 수집

## 개요
- 기상청 API(apihub.kma.go.kr)에서 해당 연도 **6월~8월** 기간의 일별 체감온도(ta_chi) 격자 데이터를 수집 (매일 14시 기준)
- 미리 저장된 격자 위경도(`latlon.pkl`)와 서울시 위치 필터(`seoul_locations.csv`)를 사용하여 **서울 지역 데이터만** 추출
- 결측값(-999.0)을 제거하고 날짜/위도/경도/체감온도를 CSV로 저장

### 입출력 파일
| 구분 | 파일 |
|------|------|
| 입력 | `../data_processed/processed/grid/latlon.pkl` (격자 위경도 데이터) |
| 입력 | `../data_processed/processed/grid/seoul_locations_500m.csv` (서울시 격자 위치 목록) |
| 출력 | `../data_processed/processed/ta_chi/ta_chi_seoul_{year}_06-08.csv` |

### 최적화 포인트
| 항목 | 기존 | 개선 |
|------|------|------|
| 정적 데이터 로드 | 루프 안에서 매번 로드 (92일 × 2회) | **1회만 로드** |
| 서울 필터링 | 매번 pandas merge | **사전 인덱스 계산 → NumPy 배열 인덱싱** |
| API 호출 | 순차 (1개씩) | **병렬 처리 (ThreadPoolExecutor)** |
| 결과 누적 | pd.concat 반복 (O(n²)) | **리스트 수집 → 1회 concat** |
| 에러 처리 | 무한 재시도, 타임아웃 없음 | **지수 백오프 재시도 (최대 3회), 30초 타임아웃** |
| 진행 추적 | print만 | **tqdm 프로그레스 바** |

## 1. 라이브러리 임포트

In [2]:
import requests
import re
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

## 2. 설정값

In [3]:
# ===== 여기에 연도와 API 인증키를 입력하세요 =====
year = 2025
authkey = "a8_vlkVwRr6P75ZFcCa-hA"  # 기상청 API 인증키

# 병렬 요청 설정
MAX_WORKERS = 5       # 동시 API 요청 수
MAX_RETRIES = 3       # 최대 재시도 횟수
REQUEST_TIMEOUT = 30  # 요청 타임아웃 (초)

## 3. 정적 데이터 사전 로드 및 서울 인덱스 계산
격자 위경도와 서울 위치 데이터를 **1회만 로드**하고, 전체 격자 중 서울에 해당하는 인덱스를 미리 계산합니다. 이후 루프에서는 NumPy 배열 인덱싱만으로 서울 데이터를 추출합니다.

In [4]:
# 격자 위경도 데이터 (1회 로드)
df_grid = pd.read_pickle("../data_processed/processed/grid/latlon.pkl")
df_seoul = pd.read_csv("../data_processed/processed/grid/seoul_locations_500m.csv")

# 서울 좌표에 해당하는 격자 인덱스를 사전 계산
# → 루프에서 매번 merge 할 필요 없이, numpy 인덱싱으로 즉시 추출
df_grid_indexed = df_grid.reset_index().rename(columns={"index": "grid_idx"})
seoul_merged = df_grid_indexed.merge(df_seoul, on=["Latitude", "Longitude"])
seoul_positions = seoul_merged["grid_idx"].values          # 전체 격자에서 서울 위치의 인덱스
seoul_lats = seoul_merged["Latitude"].values               # 서울 위도 배열
seoul_lons = seoul_merged["Longitude"].values              # 서울 경도 배열
grid_size = len(df_grid)                                   # 전체 격자 크기 (응답 검증용)

print(f"전체 격자 수: {grid_size:,}")
print(f"서울 격자 수: {len(seoul_positions):,}")
print(f"서울 격자 비율: {len(seoul_positions)/grid_size*100:.1f}%")

전체 격자 수: 4,198,401
서울 격자 수: 2,304
서울 격자 비율: 0.1%


## 4. API 요청 함수 (재시도 + 타임아웃 + 응답 검증)

In [5]:
def parse_api_response(content: bytes) -> np.ndarray:
    """API 응답을 파싱하여 float 배열로 변환"""
    text = content.decode("utf-8")[16:]
    clean = re.sub(r"[^\d\.\,\-\s]", "", text)
    parts = re.split(r"[,\s]+", clean)
    return np.array([float(x) for x in parts if x])


def fetch_one_day(date_str: str, session: requests.Session) -> tuple:
    """
    하루치 체감온도 데이터를 가져와서 서울 지역만 추출하여 반환.
    
    Returns:
        (date_str, DataFrame) 성공 시
        (date_str, None)      실패 시
    """
    url = (
        f"https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph-sfc_obs_nc_api"
        f"?obs=ta_chi&tm={date_str}1400&disp=A&authKey={authkey}"
    )
    
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            
            if resp.status_code == 200:
                ta_all = parse_api_response(resp.content)
                
                # 응답 크기 검증: 격자 수와 일치하는지 확인
                if len(ta_all) != grid_size:
                    raise ValueError(
                        f"응답 크기 불일치: 예상 {grid_size}, 수신 {len(ta_all)}"
                    )
                
                # 서울 지역만 NumPy 인덱싱으로 즉시 추출 (merge 불필요)
                ta_seoul = ta_all[seoul_positions]
                
                day_df = pd.DataFrame({
                    "DateTime": pd.Timestamp(date_str),
                    "Latitude": seoul_lats,
                    "Longitude": seoul_lons,
                    "ta_chi": ta_seoul,
                })
                # 결측값(-999.0) 제거
                day_df = day_df[day_df["ta_chi"] != -999.0]
                return (date_str, day_df)
            
            elif resp.status_code >= 500:
                last_error = f"서버 에러 {resp.status_code}"
                wait = 2 ** attempt
                time.sleep(wait)
                continue
            else:
                last_error = f"HTTP {resp.status_code}: {resp.content.decode('utf-8')[:200]}"
                break  # 4xx 에러는 재시도 불필요
                
        except requests.Timeout:
            last_error = f"타임아웃 ({REQUEST_TIMEOUT}초)"
            time.sleep(2 ** attempt)
        except requests.ConnectionError as e:
            last_error = f"연결 에러: {e}"
            time.sleep(2 ** attempt)
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            break
    
    print(f"  [실패] {date_str}: {last_error} ({attempt+1}회 시도)")
    return (date_str, None)

## 5. 병렬 수집 실행 (6월 1일 ~ 8월 31일)

In [6]:
# 수집 대상 날짜 리스트 생성
start = datetime(year, 6, 1)
end = datetime(year, 8, 31)
dates = []
current = start
while current <= end:
    dates.append(current.strftime("%Y%m%d"))
    current += timedelta(days=1)

print(f"수집 기간: {start.strftime('%Y-%m-%d')} ~ {end.strftime('%Y-%m-%d')}")
print(f"총 {len(dates)}일, 동시 요청 수: {MAX_WORKERS}")

# 병렬 수집 실행
collected = []    # 성공한 DataFrame 리스트
failed = []       # 실패한 날짜 리스트

session = requests.Session()
# Connection pooling: 동시 연결 수에 맞게 어댑터 설정
adapter = requests.adapters.HTTPAdapter(
    pool_connections=MAX_WORKERS,
    pool_maxsize=MAX_WORKERS,
)
session.mount("https://", adapter)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(fetch_one_day, d, session): d 
        for d in dates
    }
    
    with tqdm(total=len(dates), desc="API 수집", unit="일") as pbar:
        for future in as_completed(futures):
            date_str, day_df = future.result()
            if day_df is not None:
                collected.append(day_df)
            else:
                failed.append(date_str)
            pbar.update(1)

session.close()

# 날짜 순으로 정렬하여 합치기 (1회 concat)
if collected:
    result = pd.concat(collected, ignore_index=True)
    result.sort_values(["DateTime", "Latitude", "Longitude"], inplace=True)
    result.reset_index(drop=True, inplace=True)
else:
    result = pd.DataFrame(columns=["DateTime", "Latitude", "Longitude", "ta_chi"])

print(f"\n수집 완료: {len(dates) - len(failed)}/{len(dates)}일 성공, 총 {len(result):,}개 행")
if failed:
    failed.sort()
    print(f"실패한 날짜 ({len(failed)}일): {failed}")

수집 기간: 2025-06-01 ~ 2025-08-31
총 92일, 동시 요청 수: 5


API 수집:   0%|          | 0/92 [00:00<?, ?일/s]


수집 완료: 92/92일 성공, 총 211,968개 행


## 5-1. 위치별 평균 체감온도 계산
92일간 같은 위치(Latitude, Longitude)에 대해 수집된 체감온도의 **평균값**을 계산합니다.
- 약 211,968행 (2,304 위치 × 92일) → **2,304행** (위치별 1행)으로 집계

In [10]:
# 같은 위치(Latitude, Longitude)별로 92일간의 체감온도 평균 계산
result_mean = (
    result
    .groupby(["Latitude", "Longitude"], as_index=False)["ta_chi"]
    .mean()
)

result_mean["ta_chi"] = result_mean["ta_chi"].round(2)

print(f"평균 계산 전: {len(result):,}행 (위치 {result[['Latitude','Longitude']].drop_duplicates().shape[0]:,}개 × 최대 92일)")
print(f"평균 계산 후: {len(result_mean):,}행 (위치별 1행)")
print(f"ta_chi 평균 범위: {result_mean['ta_chi'].min():.2f} ~ {result_mean['ta_chi'].max():.2f}")
print(f"\n미리보기:")
result_mean.head(10)

평균 계산 전: 211,968행 (위치 2,304개 × 최대 92일)
평균 계산 후: 2,304행 (위치별 1행)
ta_chi 평균 범위: 27.07 ~ 32.65

미리보기:


,Latitude,Longitude,ta_chi
0,37.430523,127.065697,29.74
1,37.430584,127.059906,29.35
2,37.430645,127.054115,28.98
3,37.430706,127.048325,27.84
4,37.435139,127.065773,29.75
5,37.435200,127.059982,29.56
6,37.435261,127.054192,29.38
7,37.435322,127.048401,28.48
8,37.436680,126.909401,29.94
9,37.436729,126.903603,30.21


In [11]:
# 전체 행 수
total_rows = len(result_mean)

# (Latitude, Longitude) 조합의 유니크 개수
unique_coords = result_mean[["Latitude", "Longitude"]].drop_duplicates().shape[0]

print(f"전체 행 수: {total_rows}")
print(f"고유한 위경도 개수: {unique_coords}")

if total_rows == unique_coords:
    print("✅ 모든 위도/경도는 서로 다릅니다. (중복 없음)")
else:
    print("❌ 중복된 위도/경도가 존재합니다.")


전체 행 수: 2304
고유한 위경도 개수: 2304
✅ 모든 위도/경도는 서로 다릅니다. (중복 없음)


## 6. 결과 저장 및 확인

In [12]:
# 위치별 평균 결과 저장
output_path_mean = f"../data_processed/processed/ta_chi/ta_chi_seoul_mean_{year}.csv"
result_mean.to_csv(output_path_mean, index=False)

print(f"평균 결과 저장 완료: {output_path_mean}")
print(f"총 행 수: {len(result_mean):,}")
print(f"위치 수: {len(result_mean):,}")
print(f"ta_chi 평균 범위: {result_mean['ta_chi'].min():.2f} ~ {result_mean['ta_chi'].max():.2f}")
print(f"\n결과 미리보기:")
result_mean.head(10)

평균 결과 저장 완료: ../data_processed/processed/ta_chi/ta_chi_seoul_mean_2025.csv
총 행 수: 2,304
위치 수: 2,304
ta_chi 평균 범위: 27.07 ~ 32.65

결과 미리보기:


,Latitude,Longitude,ta_chi
0,37.430523,127.065697,29.74
1,37.430584,127.059906,29.35
2,37.430645,127.054115,28.98
3,37.430706,127.048325,27.84
4,37.435139,127.065773,29.75
5,37.435200,127.059982,29.56
6,37.435261,127.054192,29.38
7,37.435322,127.048401,28.48
8,37.436680,126.909401,29.94
9,37.436729,126.903603,30.21
